IMPORTS DE LAS LIBRERÍAS NECESARIAS

In [1]:
from ultralytics import YOLO # Modelo YOLO y entrenar nuestro detector de matrículas
import cv2 # Procesamiento de imágenes
from collections import defaultdict # Manejar diccionarios con listas
import csv # Manejar archivos CSV
from PIL import Image # Manejar imágenes
import torch # Manejo de tensores y modelos
from transformers import AutoProcessor, AutoModelForVision2Seq # Modelos de visión a secuencia
import numpy as np  # Manejo de arreglos numéricos
import math # Funciones matemáticas
import easyocr # Reconocimiento óptico de caracteres (OCR)


c:\Users\juanf\anaconda3\envs\VC_P4\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INTENTO DE DETECCIÓN DE MATRÍCULAS BASADA EN CONTORNOS

In [ ]:

# Carga del modelo
model = YOLO('yolo11n.pt')

filename = "C0142.mp4"
cap = cv2.VideoCapture(filename)

cv2.namedWindow('Deteccion con YOLO', cv2.WINDOW_NORMAL)
cv2.resizeWindow('Deteccion con YOLO', 1280, 720)

detections_count = 0

# funcion para detectar matrículas
def detect_plate(car_region):
    # paso la imagen a escala de grises
    gris = cv2.cvtColor(car_region, cv2.COLOR_BGR2GRAY)

    # suavizo para reducir el ruido
    gris = cv2.GaussianBlur(gris, (5, 5), 0)

    # aplico umbral adaptativo para destacar los bordes
    img_th1 = cv2.adaptiveThreshold(gris, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 5) 

    # obtengo los contornos externos
    contornos, _ = cv2.findContours(img_th1, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # comprobamos los contornos
    candidatos_mat = []
    for contorno in contornos:
        # aproximo el contorno a un polígono
        perimeter = cv2.arcLength(contorno, True)
        aprox = cv2.approxPolyDP(contorno, 0.018 * perimeter, True)

        # obtengo el rectángulo
        x, y, w, h = cv2.boundingRect(aprox)

        # características para identificar la matrícula
        aspect_ratio = w / float(h)
        area = w * h
        car_area = car_region.shape[0] * car_region.shape[1]
        relative_area = area / car_area

        if (2.0 <= aspect_ratio <= 5.5 and 
            0.01 <= relative_area <= 0.15 and
            w > 40 and h > 10):

            # Puntuación basada en qué tan cerca está del ratio ideal
            ideal_ratio = 4.5
            ratio_score = 1 - abs(aspect_ratio - ideal_ratio) / ideal_ratio
            
            candidatos_mat.append({
                'contour': aprox,
                'bbox': (x, y, w, h),
                'score': ratio_score * relative_area,
                'aspect_ratio': aspect_ratio
            })

    # cogemos el mejor candidato
    if candidatos_mat:
        mejor_matricula = max(candidatos_mat, key=lambda x: x['score'])
        return mejor_matricula

while cap.isOpened():
    ret, frame = cap.read()

    # si no hay imagen salimos
    if not ret:
        break

    # se ejecuta el modelo en el frame y se añaden los recuadros
    results = model(frame, classes=[2, 3, 5, 7], conf=0.5, verbose=False)
    annotated_frame = results[0].plot()

    # busco las matrículas de cada vehículo
    for result in results[0].boxes.data:
        x1, y1, x2, y2, conf, cls = result
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        
        # Extraer región del vehículo
        car_region = frame[y1:y2, x1:x2].copy()

        if car_region.size > 0:
            # busco la matricula
            plate = detect_plate(car_region)
            
            # si la encuentro, la dibujo
            if plate is not None:
                detections_count += 1
                px, py, pw, ph = plate['bbox']
                
                # Ajustar coordenadas al frame completo
                px += x1
                py += y1
                
                # Dibujar rectángulo de la matrícula en rojo
                cv2.rectangle(annotated_frame, (px, py), (px + pw, py + ph), (0, 0, 255), 2)
                cv2.putText(annotated_frame, 'PLATE', (px, py - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
    
    frame = annotated_frame

    cv2.imshow('Deteccion con YOLO', annotated_frame)

    # se sale con ESC o Q/q
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q') or key == ord('Q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"\nProcesamiento finalizado. Total de matrículas detectadas: {detections_count}")

CELDA DE ENTRENAMIENTO MODELO MATRÍCULAS

In [ ]:
# Descomentar para entrenar de nuevo el modelo YOLO en las matrículas

"""
from ultralytics import YOLO

# Cargar modelo preentrenado
model = YOLO("yolo11n.pt")

# Entrenar
model.train(
    data="dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolo_matriculas",
    device=0
)

# Predicción en validación
results = model.predict(source='C:/Users/juanf/Desktop/Large-License-Plate-Detection-Dataset/images/val', save=True)
"""

 CONFIGURACIÓN DEL MODELO SMOLVLM PARA OCR

In [2]:

# Establecemos el dispositivo para el modelo
device = "cuda" if torch.cuda.is_available() else "cpu"

# Cargamos el modelo SmolVLM-Instruct que es adecuado para tareas de visión a secuencia
model_name = "HuggingFaceTB/SmolVLM-Instruct"

# Cargamos el procesador asociado al modelo
processor = AutoProcessor.from_pretrained(model_name)

# Cargamos el modelo con el tipo de dato adecuado según el dispositivo
model = AutoModelForVision2Seq.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)


c:\Users\juanf\anaconda3\envs\VC_P4\lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON smolVLM

In [3]:
reader = easyocr.Reader(['es', 'en'], gpu=torch.cuda.is_available())

def extraer_texto_matricula_smolVLM(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando SmolVLM
    """

    try:
        # Extraemos la región de interés (ROI) de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        # Verificamos si la ROI está vacía
        if roi.size == 0:
            return ""

        # Convertimos de BGR (OpenCV) a RGB (PIL)
        roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        pil_image = Image.fromarray(roi_rgb)
        
        # Prompt específico para lectura de matrículas
        prompt = "Read the license plate number in this image. Only output the alphanumeric characters you see, without spaces or additional text."
        
        # Preparamos la plantilla de entrada para el modelo (imagen + prompt)
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt}
                ]
            }
        ]

        # Aplicamos la plantilla
        text = processor.apply_chat_template(messages, add_generation_prompt=True)

        # Cargamos la imagen y el texto al procesador
        inputs = processor(text=[text], images=[pil_image], return_tensors="pt")

        # Movemos los tensores al dispositivo adecuado
        inputs = inputs.to(device)
        
        # Generamos la predicción sin calcular gradientes(no es entrenamiento)
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs, # Desempaquetamos los inputs
                max_new_tokens=15, # Máximo número de tokens a generar
                do_sample=False 
            )

        # Decodificamos resultado
        generated_texts = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Extraemos solo el texto de la respuesta
        texto = generated_texts[0].split("Assistant:")[-1].strip()

        # Limpiamos el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in texto if c.isalnum()).upper()
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula: {e}")
        return ""

FUNCIÓN PARA EXTRAER TEXTO DE MATRÍCULA CON EASYOCR

In [4]:
def extraer_texto_matricula_easyOCR(frame, x1, y1, x2, y2):
    """
    Extrae el texto de una matrícula usando easyOCR - VERSION MEJORADA
    """
    try:
        # Extraer ROI de la matrícula con un pequeño padding
        padding = 5
        roi = frame[max(0, y1-padding):min(frame.shape[0], y2+padding), 
                   max(0, x1-padding):min(frame.shape[1], x2+padding)]
        
        if roi.size == 0:
            return ""
        
        # MEJORA 1: Redimensionar imagen si es muy pequeña
        height, width = roi.shape[:2]
        if height < 50 or width < 150:
            scale_factor = max(50/height, 150/width)
            new_width = int(width * scale_factor)
            new_height = int(height * scale_factor)
            roi = cv2.resize(roi, (new_width, new_height), interpolation=cv2.INTER_CUBIC)
        
        # MEJORA 2: Probar con la imagen original
        resultados_original = reader.readtext(
            roi,
            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ',
            paragraph=False,
            detail=1,
            batch_size=1
        )
        
        # MEJORA 3: Probar también con preprocesamiento
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        enhanced = clahe.apply(gray)
        _, thresh = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        resultados_procesada = reader.readtext(
            thresh,
            allowlist='0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ',
            paragraph=False,
            detail=1,
            batch_size=1
        )
        
        # Combinar resultados y elegir el mejor
        todos_resultados = resultados_original + resultados_procesada
        
        if not todos_resultados:
            return ""
        
        # Filtrar resultados con confianza muy baja
        resultados_filtrados = [r for r in todos_resultados if r[2] > 0.3]
        
        if not resultados_filtrados:
            return ""
        
        # Elegir el resultado con mayor confianza
        mejor_resultado = max(resultados_filtrados, key=lambda x: x[2])
        text = mejor_resultado[1]
        
        # Limpiar el texto: solo alfanuméricos
        texto_limpio = ''.join(c for c in text if c.isalnum()).upper()
        
        # MEJORA 4: Filtrar resultados que son demasiado cortos o largos
        if len(texto_limpio) < 4 or len(texto_limpio) > 12:
            return ""
        
        return texto_limpio
    
    except Exception as e:
        print(f"Error al procesar matrícula con EasyOCR: {e}")
        return ""
    

FUNCIÓN PARA ASOCIAR MATRÍCULAS CON VEHÍCULOS

In [5]:
def asociar_matricula_con_vehiculo(plate_box, vehicle_boxes):

    # Obtenemos el centro de la matrícula
    px1, py1, px2, py2 = plate_box
    plate_center_x = (px1 + px2) / 2
    plate_center_y = (py1 + py2) / 2
    
    # Inciamos variables auxiliares para encontrar el mejor vehículo
    mejor_vehiculo = None
    mejor_distancia = float('inf')
    
    # Recorremos las cajas de vehículos
    for idx, vbox in enumerate(vehicle_boxes):
        vx1, vy1, vx2, vy2 = vbox
        
        # Verificamos si la matrícula está dentro del vehículo( y si es así, la asociamos directamente)
        if vx1 <= plate_center_x <= vx2 and vy1 <= plate_center_y <= vy2:
            return idx
        
        # Calculamos distancia al centro del vehículo
        vcenter_x = (vx1 + vx2) / 2
        vcenter_y = (vy1 + vy2) / 2
        distancia = np.sqrt((plate_center_x - vcenter_x)**2 + (plate_center_y - vcenter_y)**2)
        
        # Actualizamos el mejor vehículo si la distancia es menor
        if distancia < mejor_distancia:
            mejor_distancia = distancia
            mejor_vehiculo = idx
    
    # Solo vamos a asociar si la distancia es razonable (menos de 200 píxeles)
    if mejor_distancia < 200:
        return mejor_vehiculo
    return None

# PROCESAMIENTO DEL VIDEO

CARGA DE MODELOS YOLO

In [6]:
# Cargamos el modelo general de detección de objetos YOLOv11
general = YOLO("yolo11n.pt")

# Cargamos el modelo específico entrenado para detección de matrículas
matriculas = YOLO("runs/detect/yolo_matriculas/weights/best.pt")

CONFIGURACIÓN DE PARÁMETROS

In [7]:

video_path = "C0142.MP4"
output_path = "detecciones_con_matriculas.mp4"
csv_path = "vehiculos_matriculas_unicas.csv"
tracker = "bytetrack.yaml"

conf_general = 0.5
conf_plate = 0.3
classes_general = [0, 2, 3, 5, 7]  # person, car, motorcycle, bus, truck

# Procesar OCR solo cada N frames
OCR_CADA_N_FRAMES = 5

# CAMBIAR ESTO para elegir qué OCR usar:
USAR_SMOLVLM = True  # True = SmolVLM, False = EasyOCR

PROCESAMIENTO

In [8]:

print("=" * 60)
print("INICIANDO PROCESAMIENTO DE VIDEO")
print("=" * 60)
print(f"Método OCR: {'SmolVLM' if USAR_SMOLVLM else 'EasyOCR'}")
print(f"Procesando OCR cada {OCR_CADA_N_FRAMES} frames")
print("=" * 60)

# Diccionario para almacenar la mejor matrícula por vehículo
# key: track_id, value: {'frame', 'bbox_vehiculo', 'bbox_matricula', 'texto', 'conf_mat', 'score', 'tipo', 'conf_obj'}
mejores_matriculas_por_vehiculo = {}

# Diccionario para almacenar info básica de cada vehículo (para el CSV final)
info_vehiculos = {}

# Tracking con el modelo general
results_stream = general.track(
    source=video_path,
    tracker=tracker,
    classes=classes_general,
    conf=conf_general,
    persist=True,
    stream=True
)

# Variables auxiliares
h, w, fps = None, None, 30
writer = None

print("\nProcesando frames...")

# Recorremos frame a frame
for frame_num, r in enumerate(results_stream):
    
    if frame_num % 50 == 0:
        print(f"Frame {frame_num}...")
    
    # Frame original y con tracking
    frame = r.orig_img.copy()
    tracked_frame = r.plot()

    # Inicializar writer
    if writer is None:
        h, w = frame.shape[:2]
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    
    # Guardar objetos trackeados del frame actual
    objetos_trackeados = []

    # Procesar cajas detectadas
    if hasattr(r, "boxes") and r.boxes is not None:
        for box in r.boxes:
            if box.id is None:
                continue

            cls = int(box.cls)
            track_id = int(box.id)
            conf = float(box.conf)
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            tipo = general.names[cls]

            # Guardar info básica del vehículo
            if track_id not in info_vehiculos:
                info_vehiculos[track_id] = {
                    'tipo': tipo,
                    'conf': conf,
                    'primer_frame': frame_num,
                    'ultimo_frame': frame_num,
                    'bbox': (x1, y1, x2, y2)
                }
            else:
                info_vehiculos[track_id]['ultimo_frame'] = frame_num
                info_vehiculos[track_id]['bbox'] = (x1, y1, x2, y2)

            objetos_trackeados.append({
                'track_id': track_id,
                'tipo': tipo,
                'bbox': (x1, y1, x2, y2)
            })
    
    # Procesar OCR solo cada N frames
    if frame_num % OCR_CADA_N_FRAMES == 0:
        
        # Detectar matrículas
        res_plate = matriculas(frame, conf=conf_plate, verbose=False)[0]

        if hasattr(res_plate, "boxes") and res_plate.boxes is not None:
            for pbox in res_plate.boxes:
                
                x1m, y1m, x2m, y2m = map(int, pbox.xyxy[0].tolist())
                conf_plate_det = float(pbox.conf)
                
                # Extraer texto según método elegido
                if USAR_SMOLVLM:
                    texto_matricula = extraer_texto_matricula_smolVLM(frame, x1m, y1m, x2m, y2m)
                else:
                    texto_matricula = extraer_texto_matricula_easyOCR(frame, x1m, y1m, x2m, y2m)

                # Solo procesar si se extrajo texto
                if not texto_matricula:
                    continue

                # Asociar matrícula con vehículos
                for obj in objetos_trackeados:
                    if obj['tipo'] not in ['car', 'motorcycle', 'bus', 'truck']:
                        continue
                    
                    x1v, y1v, x2v, y2v = obj['bbox']
                    track_id = obj['track_id']
                    
                    # Verificar intersección
                    intersecta_x = max(0, min(x2v, x2m) - max(x1v, x1m))
                    intersecta_y = max(0, min(y2v, y2m) - max(y1v, y1m))
                    area_interseccion = intersecta_x * intersecta_y
                    
                    if area_interseccion > 0:
                        # Calcular score de calidad (confianza + longitud del texto)
                        score_calidad = conf_plate_det * (len(texto_matricula) / 10.0)
                        
                        # Actualizar si es la primera detección o si es mejor
                        if track_id not in mejores_matriculas_por_vehiculo:
                            mejores_matriculas_por_vehiculo[track_id] = {
                                'frame': frame_num,
                                'bbox_vehiculo': (x1v, y1v, x2v, y2v),
                                'bbox_matricula': (x1m, y1m, x2m, y2m),
                                'texto': texto_matricula,
                                'conf_mat': conf_plate_det,
                                'score': score_calidad,
                                'tipo': obj['tipo'],
                                'conf_obj': info_vehiculos[track_id]['conf']
                            }
                        else:
                            # Solo actualizar si el nuevo score es mejor
                            if score_calidad > mejores_matriculas_por_vehiculo[track_id]['score']:
                                mejores_matriculas_por_vehiculo[track_id] = {
                                    'frame': frame_num,
                                    'bbox_vehiculo': (x1v, y1v, x2v, y2v),
                                    'bbox_matricula': (x1m, y1m, x2m, y2m),
                                    'texto': texto_matricula,
                                    'conf_mat': conf_plate_det,
                                    'score': score_calidad,
                                    'tipo': obj['tipo'],
                                    'conf_obj': info_vehiculos[track_id]['conf']
                                }
                        break

    # DIBUJAR EN EL VIDEO: Mostrar matrícula junto a cada vehículo
    annotated = tracked_frame.copy()
    
    for obj in objetos_trackeados:
        track_id = obj['track_id']
        
        # Si este vehículo tiene matrícula detectada, mostrarla
        if track_id in mejores_matriculas_por_vehiculo:
            x1v, y1v, x2v, y2v = obj['bbox']
            texto_mat = mejores_matriculas_por_vehiculo[track_id]['texto']
            
            # Dibujar texto de la matrícula arriba del vehículo
            texto = f"ID:{track_id} - {texto_mat}"
            
            # Calcular tamaño del texto
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.7
            thickness = 2
            (text_width, text_height), baseline = cv2.getTextSize(texto, font, font_scale, thickness)
            
            # Posición arriba del vehículo
            text_x = x1v
            text_y = y2v + text_height + 10
            
            # Fondo para el texto (verde)
            cv2.rectangle(annotated, 
                         (text_x, text_y - text_height - 5), 
                         (text_x + text_width + 10, text_y + 5), 
                         (0, 255, 0), -1)
            
            # Texto en blanco
            cv2.putText(annotated, texto, (text_x + 5, text_y), 
                       font, font_scale, (255, 255, 255), thickness)

    # Escribir frame
    writer.write(annotated)

# Cerrar video
if writer is not None:
    writer.release()

INICIANDO PROCESAMIENTO DE VIDEO
Método OCR: SmolVLM
Procesando OCR cada 5 frames

Procesando frames...

video 1/1 (frame 1/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 65.9ms
Frame 0...
video 1/1 (frame 2/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 29.3ms
video 1/1 (frame 3/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 31.0ms
video 1/1 (frame 4/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 32.2ms
video 1/1 (frame 5/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 15.2ms
video 1/1 (frame 6/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 18.8ms
video 1/1 (frame 7/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 21.1ms
video 1/1 (frame 8/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 18.4ms
video 1/1 (frame 9/2832) c:\Users\juanf\Desktop\VC\VC_P4\C0142.MP4: 384x640 1 car, 1 bus, 16.9ms
video 1/1 (

GENERAR CSV CON FILAS ÚNICAS (UNA POR VEHÍCULO)

In [10]:
print("\nGenerando CSV con filas únicas por vehículo...")

with open(csv_path, 'w', newline='', encoding='utf-8') as f_csv:
    csv_writer = csv.writer(f_csv)
    
    # Encabezados
    csv_writer.writerow([
        'track_id',
        'tipo_vehiculo', 
        'conf_vehiculo',
        'frame_primera_aparicion',
        'frame_ultima_aparicion',
        'matricula_detectada',
        'texto_matricula',
        'conf_matricula',
        'frame_mejor_deteccion',
        'vehiculo_x1', 'vehiculo_y1', 'vehiculo_x2', 'vehiculo_y2',
        'matricula_x1', 'matricula_y1', 'matricula_x2', 'matricula_y2'
    ])
    
    # Escribir una fila por cada vehículo
    vehiculos_con_matricula = 0
    vehiculos_sin_matricula = 0
    
    for track_id in sorted(info_vehiculos.keys()):
        info = info_vehiculos[track_id]
        
        # Si tiene matrícula
        if track_id in mejores_matriculas_por_vehiculo:
            mat = mejores_matriculas_por_vehiculo[track_id]
            vehiculos_con_matricula += 1
            
            csv_writer.writerow([
                track_id,
                mat['tipo'],
                f"{mat['conf_obj']:.2f}",
                info['primer_frame'],
                info['ultimo_frame'],
                'Si',
                mat['texto'],
                f"{mat['conf_mat']:.2f}",
                mat['frame'],
                *mat['bbox_vehiculo'],
                *mat['bbox_matricula']
            ])
        else:
            # Sin matrícula
            vehiculos_sin_matricula += 1
            
            csv_writer.writerow([
                track_id,
                info['tipo'],
                f"{info['conf']:.2f}",
                info['primer_frame'],
                info['ultimo_frame'],
                'No',
                '', '', '',
                *info['bbox'],
                '', '', '', ''
            ])

print(f"\n✅ CSV generado: {csv_path}")
print("\n" + "=" * 60)
print("RESUMEN")
print("=" * 60)
print(f"Total de vehículos detectados: {len(info_vehiculos)}")
print(f"Vehículos CON matrícula: {vehiculos_con_matricula}")
print(f"Vehículos SIN matrícula: {vehiculos_sin_matricula}")
print(f"Tasa de detección: {vehiculos_con_matricula/len(info_vehiculos)*100:.1f}%")
print("=" * 60)


Generando CSV con filas únicas por vehículo...

✅ CSV generado: detecciones_tracking_matriculas.csv

RESUMEN
Total de vehículos detectados: 152
Vehículos CON matrícula: 26
Vehículos SIN matrícula: 126
Tasa de detección: 17.1%
